# Polypeptide Toxicity Classification

Don Frenzel, Rohan Castillo, Lasaro Morell

Binary classification of peptides as **toxic (label 1)** or **non-toxic (label 0)** from
sequence alone. The dataset is 6,387 training and 1,126 test peptides of length 12-51,
split roughly 71% / 29% non-toxic to toxic in both files.

The experiment covers three independent feature representations, each with its own
classifiers, so that the contribution of the representation can be separated from the
contribution of the model:

| Section | Representation | Classifiers |
|---|---|---|
| 1 | Physicochemical property matrix, summarised per sequence | SVM, Random Forest, AdaBoost |
| 2 | Sequence-based: bigram matrix, occurrence, amino acid composition | CNN, AdaBoost |
| 3 | ProtBert protein language model embeddings | KNN |

Every model is scored with accuracy, sensitivity, specificity and MCC, on 5-fold
cross-validation and on the held-out test set.

**Running this notebook:** `uv run jupyter lab` from the project root, then pick the
project kernel. Paths are relative to the project root, so keep the working directory
there. Run the Setup cell first, then any section top to bottom.

## Setup

Every import used anywhere in the notebook, plus `fastaConverter`, which all three
sections use to read the FASTA files. Run this cell once before any section.

In [ ]:
#----------------IMPORTS----------------#

###Data handling and plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

###Biopython, for reading FASTA files
from Bio import SeqIO

###PyTorch, for the CNN in section 2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset
from torchmetrics.classification import BinaryMatthewsCorrCoef
from torchmetrics.classification import MulticlassConfusionMatrix

###Protein language model, for section 3
from transformers import BertModel, BertTokenizerFast
from tqdm import tqdm

###ML models
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

###Model selection, preprocessing and evaluation
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import make_scorer, accuracy_score, confusion_matrix, classification_report, matthews_corrcoef


#----------------FASTA LOADING----------------#

### NOTE LABEL '1' is toxic, LABEL '0' is nontoxic.
# Reads FASTA file and converts to datafram with columns: label, sequence, length
def fastaConverter(filename):
    dataList = []

    with open(filename, "r") as handle:
        for entry in SeqIO.parse(handle, "fasta"):
            dataList.append({
                "label": int(entry.id[8]),
                "sequence": str(entry.seq),
                "length": len(entry.seq)
            })

    return pd.DataFrame(dataList)


---
## 1. Physicochemical features

Each amino acid carries numerical values for physical and chemical properties
(hydrophobicity, charge, polarity and so on), read here from
`data/physiochemical_attributes.xlsx`. A peptide becomes a matrix of
`length x n_properties`, which is then collapsed to a fixed-length vector so that
sequences of different lengths can be fed to the same classifier.

### 1.1 Amino acid property lookup

Builds a dictionary mapping each amino acid letter to its vector of property values.

In [ ]:
# reads amino acid features from an excel_file and returns dictionary mapping amino acid to its features
def amino_acid_features(file_name, sheet_name):
    df = pd.read_excel(file_name, sheet_name, skiprows=1)
    df = df.set_index(df.columns[1])
    df = df.drop(columns=[df.columns[0]])
    df = df.T

    aa_dict = {}

    for aa in df.index:
        aa_dict[aa] = df.loc[aa].values

    return aa_dict

### 1.2 From sequence to fixed-length vector

A sequence becomes a matrix of per-residue property vectors, then that matrix is
summarised by concatenating the mean, standard deviation, minimum, and maximum
for each property, followed by the sequence length.

In [ ]:
# converts a sequence to a matrix: each amino acid -> feature vector
def sequence_to_matrix(sequence, aa_dict):
    return np.array([aa_dict[aa] for aa in sequence])


# converts a matrix to a feature vector containing mean, std, min, and max for each feature across the sequence 
def matrix_to_mean_std_min_max(matrix):
    mean = matrix.mean(axis=0)
    std = matrix.std(axis=0)
    min_val = matrix.min(axis=0)
    max_val = matrix.max(axis=0)

    return np.concatenate([mean, std, min_val, max_val])

In [ ]:
# adds physiochemical feature vectors to the dataframe
def add_physiochemical_features(dataframe, aa_dict):

    # convert each sequence into a matrix
    matrices = dataframe["sequence"].apply(
        lambda seq: sequence_to_matrix(seq, aa_dict)
    )

    dataframe["features"] = matrices.apply(matrix_to_mean_std_min_max)
    dataframe["features"] = dataframe.apply(
        lambda row: np.append(row["features"], row["length"]),
        axis=1
    )

    return dataframe


### 1.3 Hyperparameter search

One grid search per classifier, each using 5-fold stratified cross-validation so every
fold keeps the class ratio of the full set. Hyperparameters are selected by MCC. The SVM is wrapped in a `Pipeline` with
`StandardScaler` so that scaling is fitted inside each fold rather than on the whole
training set, which would leak test-fold statistics into training.

In [ ]:
# does grid search with cross validation for any estimator
def tune_model(estimator, param_grid, X_train, y_train):
    # stratified ensures class balance in fold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=44)

    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring=make_scorer(matthews_corrcoef), # metric used to pick best model
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    return grid

In [ ]:
# pipeline to avoid data leakage 
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", random_state=44))
])

search_spaces = {
    "SVM": (
        svm_pipeline,
        {
            "svm__C": [0.1, 1, 5, 10],
            "svm__gamma": [0.001, 0.01, 0.05, 0.1],
            "svm__class_weight": [None, "balanced"]
        }
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=44),
        {
            "n_estimators": [10, 50, 100, 200, 400],
            "max_depth": [None, 10, 20],
            "min_samples_split": [2, 5],
            "min_samples_leaf": [1, 2, 4]
        }
    ),
    "AdaBoost": (
        AdaBoostClassifier(
            estimator=DecisionTreeClassifier(random_state=44),
            random_state=44
        ),
        {
            "n_estimators": [10, 50, 100, 200, 400],
            "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
            "estimator__max_depth": [1, 2, 3],
            "estimator__min_samples_leaf": [1, 2, 4]
        }
    ),
}

### 1.4 Scoring

Refits the chosen model on the full training set and reports the four metrics on the
independent test set. Sensitivity is recall on the toxic class, specificity is recall on
the non-toxic class, and MCC is the metric to watch given the class imbalance.

In [ ]:
# models arrive already fitted: GridSearchCV refits the best estimator on the
# full training set
def evaluate_model(X_test, y_test, model):
    y_pred = model.predict(X_test)

    # confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    TN, FP, FN, TP = cm.ravel()

    accuracy = accuracy_score(y_test, y_pred)
    sensitivity = TP / (TP + FN)
    specificity = TN / (TN + FP)
    mcc = matthews_corrcoef(y_test, y_pred)

    return cm, accuracy, sensitivity, specificity, mcc

### 1.5 Run: load and featurise

Uses the mean, standard deviation, minimum, maximum, and sequence length.

In [ ]:
aa_dict = amino_acid_features("data/physiochemical_attributes.xlsx", "Sheet2")

# load training and test data
trainingData = fastaConverter("data/Train.fasta")
testingData = fastaConverter("data/test.fasta")


# convert sequences to features
trainingData = add_physiochemical_features(
    trainingData,
    aa_dict
)
testingData = add_physiochemical_features(
    testingData,
    aa_dict
)

# convert to numpy arrays
X_train = np.vstack(trainingData["features"].values)
y_train = trainingData["label"].values

X_test = np.vstack(testingData["features"].values)
y_test = testingData["label"].values

### 1.6 Run: grid search

In [ ]:
grids = {
    name: tune_model(estimator, param_grid, X_train, y_train)
    for name, (estimator, param_grid) in search_spaces.items()
}

# print best parameters and cv performance
for name, grid in grids.items():
    print(f"Best {name} Params:")
    print(grid.best_params_)
    print(f"Best {name} CV MCC: {grid.best_score_:.4f}\n")

### 1.7 Run: test set results

In [ ]:
# evaluate all models
for name, grid in grids.items():
    cm, accuracy, sensitivity, specificity, mcc = evaluate_model(
        X_test, y_test, grid.best_estimator_)

    print(f"\nFinal {name} Test Results")
    print("Confusion Matrix:")
    print(cm)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"MCC: {mcc:.4f}")

### 1.8 Recorded results

From `results/physiochemical_features__classifier_results.txt` (previous accuracy-selected
run; rerun the searches and evaluation to obtain MCC-selected results):

| Model | CV accuracy | Test accuracy | Sensitivity | Specificity | MCC |
|---|---|---|---|---|---|
| SVM | 0.9032 | 0.8970 | 0.7625 | 0.9504 | 0.7404 |
| Random Forest | 0.9061 | 0.9005 | 0.7781 | 0.9491 | 0.7501 |
| AdaBoost | 0.9023 | 0.9041 | 0.8125 | 0.9404 | 0.7618 |

All three land near 0.90 accuracy, and all three are far better at recognising
non-toxic peptides than toxic ones: specificity sits around 0.94-0.95 while sensitivity
reaches only 0.76-0.81. These recorded grid searches selected on accuracy, which on a
71/29 split rewards the majority class.

---
## 2. Sequence-based features

These features use only the ordering and combination of letters, with no reference to
the chemical nature of the amino acids:

- **Amino acid composition (AAC)** - count of each of the 20 amino acids divided by sequence length.
- **Occurrence** - count of each amino acid in the sequence.
- **Bigram** - frequency of each adjacent amino acid pair, giving a 20x20 matrix per peptide.

The bigram matrix is the reason a CNN is appropriate here: it is a genuine 2-D input
where neighbouring cells are related, so convolution over it is meaningful.

### 2.1 The CNN

Two convolutional blocks with batch normalisation, then max pooling, then two fully
connected layers, ending in two output units for the two classes. Dropout appears twice,
lightly after the convolutions and heavily before the output, to limit overfitting.

In [ ]:
class convNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        ###define first convolution, initial input layer is 1.  3 by 3 windows.  256 Filters.s
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
        ###Batch normalization recenters and rescales data
        self.bn1 = nn.BatchNorm2d(64)
        ###Second convolutional layer.  128 Filters from 64
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        ###Pooling layer, pools into several pooled maps.  
        self.pool1 = nn.MaxPool2d(2,2)

        ###Dropout basically is the chance that any given value will be dropped out of the data.  
        self.dropout_spatial = nn.Dropout2d(0.2)

        ### Linear Layer; linears are fully connected and perform final classification
        self.fc1 = nn.Linear(128 * 10 * 10, 512)
        ###Batch normalization
        self.bn_fc1 = nn.BatchNorm1d(512)
        ### Second linear layer.  
        self.fc2 = nn.Linear(512, 128)

        ###Larger dropout before output.  
        self.dropout = nn.Dropout(p=0.5)

        ###Add linear output layer; needs to end in 2 values.
        self.out = nn.Linear(in_features=128, out_features=2)

    ###Forward Pass makes sure that it can go through
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))##runs batch through first convolution.
        x = self.pool1(F.relu(self.bn2(self.conv2(x)))) ###'' second convolution
        x = self.dropout_spatial(x)

        ###Flattening layer; basically takes the whole thing and just makes it a singular 1 dimensional for compatibility with the full 
        x = torch.flatten(x, 1)

        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return x

### 2.2 Device and data

Parsing is done with Biopython's `SeqIO` rather than a plain text split, because the
earlier fastaframes approach did not separate the columns correctly.


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Running on: ',device)

In [ ]:
###Load data in, training/testing.  Split later for validation.  
trainingData = fastaConverter('data/Train.fasta')
testingData = fastaConverter('data/test.fasta')

###Important for functions, essentially acts as a library for amino acids.  
allColumns = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']

###Testing purposes only.  
sequences = trainingData.sequence


### 2.3 Amino acid composition and occurrence

`getAAC` normalises each amino acid count by sequence length. `getOCC` was intended to
piggyback on it and return the **pure counts** of each amino acid; as written it records
binary presence instead, so the magnitudes are lost.


In [ ]:
allColumns = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']

In [ ]:
def getAAC(sequence,desiredColumns):
    seqList = list(sequence)
    seqLen = len(sequence)
    counts = Counter(seqList)  ##outputs as dict type.  Convert to tuples and sort by key.

    ##normalize the data and then make it a dict at the same time.  
    counts = {key: count / seqLen for key, count in counts.items()}

    ## puts the counts data into a new dataframe and reindexes by the desired columns. 
    countsDF = pd.DataFrame([counts]).reindex(columns=desiredColumns)
    countsDF = countsDF.fillna(0).astype(int) #Handles NaN data.  

    return countsDF

In [ ]:
def getOCC(sequence,desiredColumns):
    occDict = {}
    seqList = list(sequence)

    ### does basic bucketing; if no bucket exists, creates a bucket, but only if in sequence.  Counts binary wise.  
    for aa in seqList:
        if aa not in occDict:
            occDict[aa]=1
    ###now convert to a dataframe
    occDF = pd.DataFrame([occDict]).reindex(columns=desiredColumns)
    occDF = occDF.fillna(0).astype(int) #Handles NaN values.  
    return occDF

In [ ]:
def OCCSeqs(data, desiredColumns):
    ###Def list for concatenation
    oSeqs = []
    ###
    for seq in data.sequence:
        convSeq = getOCC(seq, desiredColumns)
        oSeqs.append(convSeq)

    oSeqsOutput = pd.concat(oSeqs, ignore_index=True)
    ###rets dataframe of converted sequences.  
    return oSeqsOutput

def AACSeqs(data, desiredColumns):
    aSeqs = []
    for seq in data.sequence:
        convSeq = getAAC(seq, desiredColumns)
        aSeqs.append(convSeq)
    aSeqsOutput = pd.concat(aSeqs, ignore_index=True)
    return aSeqsOutput

### 2.4 Bigram decomposition

For a sequence of length L there are L-1 adjacent pairs. Each pair increments one cell
of a 20x20 matrix, and the matrix is divided by the pair count to normalise it. The
stack of matrices gets a trailing channel dimension so it can be fed to the CNN.

**TODO:** `bigramSeqs` and `getBigram` are still separate and could be merged.


In [ ]:
def getBigram(sequence,desiredColumns):
    ###Get bi-grams present in sequence.
    aas = 'ACDEFGHIKLMNPQRSTVWY'
    
    ##Gets all aas as a dictionary.  
    aasDict = {aa: i for i, aa in enumerate(aas)}

    ##list of all possible bigrams
    retMatrix = np.zeros((20,20))

    ### loop through in range of the sequence-1 (num of bigram pairs that can exist) and then increase the row, column value which is of the pair in the matrix by 1(for the sake of count)
    ### total number of permutations given by the sequence.  seq is of length 28, zB, so the number of pairs is 27, as the number of bigrams is always L-1 for a set of chars L.
    ### it grabs the row and column values, row being i and column being the next i to form a pair, which it then searches the value of in aas dict to get the row/column values for.  
    for i in range(len(sequence)-1):
        row = aasDict[sequence[i]]
        column = aasDict[sequence[i+1]]
        retMatrix[row,column]+=1

    totalPairs =  len(sequence)-1      

    ### Divide all vals of the retMatrix by the total pairs to normalize it.  Creates good nparray.  
    retMatrix = retMatrix/totalPairs

    ##return matrix as a numpy dataframe with the labels intact.  
    #retDF = pd.DataFrame(retMatrix, columns = desiredColumns, index = desiredColumns)
    return retMatrix

In [ ]:
def bigramSeqs(data, desiredColumns):
    cSeqs = []
    for seq in data.sequence:
        #convSeq is returned as an nparray.
        convSeq = getBigram(seq, desiredColumns)
        ###works properly, at least returns.  
        cSeqs.append(convSeq)
    ###results in stack of size (n, m, m) for n mxm matrices, channel depth must be explicit.  
    result = np.stack(cSeqs,axis=0)

    ###Adds the extra 1 dimension necessary for CNN to work.  
    result = np.expand_dims(result, axis=-1) 
    return result

### 2.5 Non-standard residue check

The 20-letter alphabet above would silently drop non-standard residues (U, X, Z), so
check whether any appear before relying on it.

In [ ]:
def checkUXZ(data):
    countU = 0
    countX = 0
    countZ= 0
    ###Tests ocurrence of u, x, and z and totals them.  
    for i, seq in enumerate(data):
        occurrence = getOCC(seq, allColumns)
        if 'u' in occurrence.columns:
            countU+=1
        if 'x' in occurrence.columns:
            countX+=1
        if 'z' in occurrence.columns:
            countZ+=1
    ###Print out the counts.
    print(countU,countX,countZ)

In [ ]:
print('Counts respectively of U, X, & Z amino acids in training and testing data:')
checkUXZ(sequences)
checkUXZ(testingData.sequence)

### 2.6 CNN training and evaluation

Splits the training set 80/20 into train and validation, trains with cross-entropy loss
and SGD with momentum, weight decay and a step learning-rate schedule, then evaluates on
the test set and plots accuracy per epoch.

In [ ]:
def CNN(trainingData, testingData, numEpochs):
    ### Now, convert all sequences to biGramSeqs and then all labels to np vectors; make sure dimensions are good to work with tensorflow.  Since data is
    ### autonormalized by function pipeline, there's no need to renormalize it.  
    classification = ['Nontoxic','Toxic']
    batchSize = 8
    
    xTrain = bigramSeqs(trainingData, allColumns)
    xTest = bigramSeqs(testingData, allColumns)

    yTrain = (trainingData.label).to_numpy()
    yTrain = np.expand_dims(yTrain, axis=-1)#expands dim by 1 to represent channel depth

    yTest = (testingData.label).to_numpy()
    yTest = np.expand_dims(yTest,axis=-1)#expands dim by 1 to represent channel depth

    ###Need to pair training and test with their labels.
    ###convert to tensors.
    xTrainTensor =torch.from_numpy(xTrain)
    yTrainTensor = torch.from_numpy(yTrain)
    xTestTensor = torch.from_numpy(xTest)
    yTestTensor = torch.from_numpy(yTest)

    ###Creates the training and test Set
    trainSet = TensorDataset(xTrainTensor, yTrainTensor)
    testSet = TensorDataset(xTestTensor, yTestTensor)
    
    ###Split training set into training and validation.  Make validation roughly same amount in training data as test data.  80/20 split.  
    trainSet, valSet = torch.utils.data.random_split(trainSet, [5110, 1277])

    ###Loaders load the data into proper datasets for the neural network to learn off of.  
    trainLoader = torch.utils.data.DataLoader(trainSet, batch_size = batchSize, shuffle = True, num_workers = 0)
    valLoader = torch.utils.data.DataLoader(valSet, batch_size = batchSize, shuffle = True, num_workers = 0)
    testLoader = torch.utils.data.DataLoader(testSet, batch_size = batchSize, shuffle = True, num_workers = 0)

    ###Defines the net for use and also prints it to the device 
    net = convNeuralNet()
    print(net.to(device))
    
    ###Def loss function and optimizer.  Using Cross-Entropy Loss and Stochastic Gradient Descent optimization. Weight decay added also to make sure to avoid overfititng. 
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  
    optimizer = optim.SGD(net.parameters(), lr=0.001, momentum = 0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5) 

    ###Training loop:
    epochs = numEpochs

    ###Track Accuracy and Loss over epochs
    accVOverEpochs = []
    accTOverEpochs = []
    ###Training/Validation loop
    for epochIndex in range(epochs):
        print("Epoch :",epochIndex+1)

        
        net.train(True) #set training mode
        ###Reset epoch values
        running_loss = 0.0
        running_accuracy = 0.0
        avgAccTOverEpoch = []
        epochTotal = 0

        #iterate over dataloader to train the data in the epoch.
        for batch_index, data in enumerate(trainLoader):
            inputs, labels = data[0].to(device), data[1].to(device)
            inputs = inputs.permute(0, 3, 1, 2).float() ###NECESSARY; MUST BE REORDERED AS SUCH TO AVOID ERROR
            labels = labels.view(-1).long() # Ensures proper shape to avoid error. 

            #reset gradients.
            optimizer.zero_grad() 

            #shape: [batch_size, 2] grab highest value and look at index
            outputs = net(inputs)

            ###Gets the sum of the correct outputs & keep tabs on the general accuracy.  
            correct = torch.sum(labels == torch.argmax(outputs, dim=1)).item() 
            running_accuracy += correct/batchSize 

            ### Calculate loss function and calculates loss value
            loss = criterion(outputs, labels) 
            running_loss += loss.item()

            ###backpropagates to learn model and then takes an optimizer's step
            loss.backward() 
            optimizer.step()

            epochTotal += labels.size(0)

            ###Checks when the batches reach 200 so that it can print regular updates on the batch loss and batch accuracy within the epoch.  
            if batch_index % 200 == 199:
                avgLossOverBatches = running_loss/200
                avgAccOverBatches = (running_accuracy/200)*100
                print(f'Batch {batch_index+1}, Loss: {avgLossOverBatches:.3f}, Accuracy: {avgAccOverBatches:.1f}%')
                avgAccTOverEpoch.append(avgAccOverBatches)

                ###reset running accuracy and loss back to zero for start of next batch
                running_accuracy = 0.0
                running_loss = 0.0
        ###Gets the average sum of the batch accuracies so that it acts as the epoch's general accuracy for the charting at the end.  
        accTAvg = sum(avgAccTOverEpoch)/len(avgAccTOverEpoch)
        accTOverEpochs.append(accTAvg)
        
        print()

        ###Validation step:  Very similar to training data
        net.train(False)  ### Disable training.  
        runningLossV = 0.0
        runningAccuracyV = 0.0

        ###Load through data in the same way tothe 
        for i, data in enumerate(valLoader):
            inputsV, labelsV = data[0].to(device), data[1].to(device)
            inputsV = inputsV.permute(0, 3, 1, 2).float()
            labelsV = labelsV.view(-1).long() 

            ###Without calculating the gradient; saves memory.  Then grabs the accuracy values for the validation set by running them through that epoch's model version.  
            with torch.no_grad():
                outputsV = net(inputsV)

                ###Records num correct, running accuracy, the loss, and the running loss for the validation set.  
                correctV = torch.sum(labelsV == torch.argmax(outputsV, dim=1)).item()
                runningAccuracyV += correctV/batchSize
                lossV = criterion(outputsV, labelsV)
                runningLossV = lossV.item()
        ###Gets the average loss and accuracy over the batches   .    
        avgLossVOverBatches = runningLossV/len(valLoader)
        avgAccVOverBatches = (runningAccuracyV/len(valLoader))*100

        ###Appends the avg accuracy for the validation set to the list of them for data viz later.  
        accVOverEpochs.append(avgAccVOverBatches)
        scheduler.step() ###advance the scheduler

        print(f'Validation Loss: {avgLossVOverBatches:.3f}, Validation Accuracy: {avgAccVOverBatches:.1f}%')


        print('*****************************************************')
        print()
        
    
    print("TRAINING COMPLETE!!!")

    print("BEGIN TESTING\n")

    ###Start testing; gets number of correct classifications and total classifications.  Resets correct var.  
    correct = 0
    total = 0
    ###Matthews Correlation Coefficient Function from PyTorch -- implemented because other group members using + good estimate.  
    mcc = BinaryMatthewsCorrCoef()
    confusMatrix = MulticlassConfusionMatrix(num_classes=3)
    
    #### since we're not training, we don't need to calculate the gradients for our outputs
    with torch.no_grad():
        for i, data in enumerate(testLoader):
            seq, labels = data[0],data[1]
            seq = seq.permute(0, 3, 1, 2).float() 
            #### calculate outputs by running images through the network net()
            outputs = net(seq)
            #### the class with the highest energy is what we choose as prediction
            _, predicted = torch.max(outputs, 1)
            targets = labels.squeeze() #compress labels so that they work with MCC.  
            
            total += labels.size(0)
            correct += (predicted.flatten() == labels.flatten()).sum().item()
            
            mcc.update(predicted, targets)
            confusMatrix.update(predicted, targets)


    ###Gets total test accuracy
    testAcc = 100* correct // total
    completeMCC = mcc.compute()
    completeCM = confusMatrix.compute()

    ###Test prints 
    print(f"Final Correct: {correct}")
    print(f"Final Total: {total}")
    
    print(f"Label Shape: {targets.shape}")
    print(f"Predicted Shape: {predicted.shape}")

    ###Prints total accuracy that will be present later on graph.  
    print(f'Accuracy of the network on the test set: {testAcc} %')
    print(f"Matthews Correlation Coefficient for Convolutional Neural Network: {completeMCC.item():.4f}")
    print(f'Confusion Matrix for Convolutional Neural Network: {completeCM}')
    
    ###Now that it's all done and over, grab the data from the epochs and plot them like before:
    plt.plot(accTOverEpochs)
    plt.plot(accVOverEpochs)
    plt.axhline(y=testAcc, color='gold', linestyle='--')
    plt.ylim(50, 100)
    plt.title('Model Accuracy, CNN')
    plt.ylabel('Accuracy [%]')
    plt.xlabel('Epochs')
    plt.legend(['Train','Val','Test'],loc='lower right')
    plt.show()

### 2.7 AdaBoost on occurrence and composition

A `ColumnTransformer` builds both feature blocks from the raw sequence column and
concatenates them, so the whole thing is a single fitted pipeline.

In [ ]:
def AdaptiveBoostingModel(trainingData, testingData):
    ### Def the columns in use/USE STANDARD COLS; NONSTANDARD NONPRESENT
    allColumns = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']

    ###Define training and testing values:
    xTrain = trainingData[['sequence']]
    yTrain = trainingData['label']

    xTest = testingData[['sequence']]
    yTest = testingData['label']

    ###USE OCCSeqs() and AACSeqs()
    ###Use these to construct the pipeline for both Ocurrence (getOCC) and for Amino Acid Composition (AAC)
    preprocessor = ColumnTransformer(transformers = [
        ('OCC', FunctionTransformer(OCCSeqs, kw_args={'desiredColumns': allColumns}),['sequence']),
        ('AAC', FunctionTransformer(AACSeqs, kw_args={'desiredColumns': allColumns}),['sequence'])])

    ###Define the AdaBoost Pipeline for output; note to self, clf just means classifier.  Bad acronym.
    adaBoostPipeline = Pipeline(steps=[('prep', preprocessor),('clf',AdaBoostClassifier())])

    ###Train the classifier model
    adaBoostPipeline.fit(xTrain, yTrain)

    ### Test it against the test set.  
    yPredicted = adaBoostPipeline.predict(xTest)

    ###Try k-fold cross validation too;
    crossValScores = cross_val_score(adaBoostPipeline, xTrain, yTrain, cv=5)

    crossValAvgScore = crossValScores.mean()

    ###MCC imp.
    mcc = matthews_corrcoef(yTest, yPredicted)

    ###Confusion Matrix Imp
    confusionMatrix = confusion_matrix(yTest, yPredicted)

    ###Print various scores.
    accuracyScore = accuracy_score(yTest, yPredicted)
    reportCard = classification_report(yTest,yPredicted)
    print("\nAccuracy: ", accuracyScore)
    print("\nFULL REPORT:\n",reportCard)
    print("Cross Validation Score: ", crossValAvgScore)
    print("Matthews Correlation Coefficient: ", mcc)
    print("Confusion Matrix: \n", confusionMatrix)

### 2.8 Run

The original script offered this as a text menu for the presentation demo; `input()`
works in a notebook, so it is kept as written. To skip the prompt, call
`CNN(trainingData, testingData, 10)` or `AdaptiveBoostingModel(trainingData, testingData)`
directly instead.

In [ ]:
userDecision = input("\nWhich would you like to run?  \n(0)CNN on BiGram Decomposition \n(1)AdaBoost on Occurrence and Amino Acid Composition?\n")
userDecision = int(userDecision)
if userDecision == 0:
    epochs = 10
    CNN(trainingData, testingData, epochs)
if userDecision == 1:
    AdaptiveBoostingModel(trainingData, testingData)

### 2.9 Recorded results

From `results/sequenceBasedFeaturesOUTPUT.txt`:

| Model | Features | Test accuracy | MCC |
|---|---|---|---|
| CNN | Bigram matrix | 92% | 0.7998 |
| AdaBoost | Occurrence + AAC | 0.8037 | 0.4952 |

The CNN on bigrams is the strongest model in the whole experiment, which suggests that
adjacency information carries real signal beyond composition. The AdaBoost result is the
weakest of any model here by a wide margin.

The training curve is saved at `results/sequenceBasedFeaturesOUTPUTCNN.png`.

---
## 3. Protein language model embeddings

A protein language model is trained on large protein databases and produces a numerical
encoding of a sequence that can be used directly as a feature vector. This section uses
`Rostlab/prot_bert`, a BERT model trained on protein sequences, and classifies its
embeddings with KNN.

Unlike the previous two sections, no features are hand-designed here: the representation
is entirely learned, so this is the test of whether a pretrained model beats explicit
feature engineering on this task.

**Note:** this section downloads model weights on first run (a few hundred MB) and will
use a GPU if one is available.

### 3.1 Reading the data

In [ ]:
trainingData = fastaConverter('data/Train.fasta')
testingData = fastaConverter('data/test.fasta')

### 3.2 Loading ProtBert and extracting embeddings

ProtBert on HuggingFace expects residues separated by spaces, so `ACDEF` has to be
rewritten as `A C D E F`. Sequences are processed in batches, and the hidden state at
index 0, the `[CLS]` token, is taken as the summary vector for the whole peptide.

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert").to('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()

In [ ]:
def proteinbert_feature_extractor(peptides, batch_size=32):
      features = []
      for i in tqdm(range(0, len(peptides), batch_size)):
        batch_seqs = peptides[i : i + batch_size]
        
        # Format: "ACDEF" -> "A C D E F", necessary for pbert huggingface
        spaced_seqs = [" ".join(list(seq)) for seq in batch_seqs]
        
        # Tokenize
        inputs = tokenizer(spaced_seqs, return_tensors='pt', padding=True, truncation=True, max_length=52)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            # Use the index 0 as summary
            batch_features = outputs.last_hidden_state[:, 0, :].cpu()
            features.append(batch_features)
            
      return torch.cat(features, dim=0)

### 3.3 KNN on the embeddings

Grid searches the neighbour count with 5-fold cross-validation, using distance weighting
so that closer neighbours count for more.

In [ ]:
def knn_plm(data):
    peptide_features = proteinbert_feature_extractor(data['sequence'])
    pep_labels = torch.tensor(data['label'].values).float().unsqueeze(1)
    X = peptide_features.numpy() 
    y = data['label'].values

    # 2. Split into training and testing (80/20 split)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. K-fold validation KNN
    grid_search = GridSearchCV(KNeighborsClassifier(weights='distance'), {'n_neighbors': range(1, (int(6387**(1/2))))}, cv=5)
    grid_search.fit(X_train, y_train)

    # 4. Evaluate
    y_pred = grid_search.predict(X_test)
    print(f"Best k param: {grid_search.best_params_} best score: {grid_search.best_score_}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(classification_report(y_test, y_pred))
    print(f"Matthews correlation coefficient(MCC): {matthews_corrcoef(y_test, y_pred)}")

### 3.4 Run

In [ ]:
knn_plm(trainingData)

### 3.5 Results

No saved output for this section yet.

---
## 4. Discussion

Collecting every recorded test-set number in one place:

| Section | Representation | Model | Accuracy | Sensitivity | Specificity | MCC |
|---|---|---|---|---|---|---|
| 2 | Bigram matrix | CNN | 0.92 | - | - | 0.7998 |
| 1 | Physicochemical | AdaBoost | 0.9041 | 0.8125 | 0.9404 | 0.7618 |
| 1 | Physicochemical | Random Forest | 0.9005 | 0.7781 | 0.9491 | 0.7501 |
| 1 | Physicochemical | SVM | 0.8970 | 0.7625 | 0.9504 | 0.7404 |
| 2 | Occurrence + AAC | AdaBoost | 0.8037 | 0.5688 | 0.8970 | 0.4952 |
| 3 | ProtBert | KNN | - | - | - | - |

Points worth developing in the report:

1. **Adjacency beats composition.** The bigram CNN is the best model and the
   occurrence/AAC AdaBoost is the worst, although both are sequence-based and use no
   chemical information. The difference between them is that bigrams preserve ordering.

2. **Physicochemical features are close behind with much less machinery.** Three
   conventional classifiers all reach roughly 0.90 on a vector of summary statistics,
   within a couple of points of the CNN.

3. **Every model is weaker on the toxic class.** Sensitivity trails specificity by 13 to
   18 points throughout. With a 71/29 split and grid searches that select on accuracy,
   this is the expected outcome and is worth addressing directly rather than reporting
   as a curiosity.

4. **The PLM section is not yet comparable.** It evaluates on an 80/20 split of the
   training data rather than on the independent test set, so its numbers cannot be put
   in the table above as they stand.